In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import glob
from pathlib import Path
import matplotlib.dates as mdates
import calendar
import numpy as np
import matplotlib.cm as cm
from adjustText import adjust_text

In [2]:
# -------- Paths --------
gauges_csv = Path("./gauges_in_basins.csv")
UYPRECIP_DIR=Path("./precip_uy")

In [3]:
df = pd.read_csv(gauges_csv)
df

,lat,lon,Estacion,Departamento,Tipo,Adm,geometry,index_right,basin_source
0,-33.46,-55.63,Capilla del Sauce,Florida,Convencional,INUMET,POINT (-55.63 -33.46),0.0,durazno
1,-32.51,-57.16,Cañada Grande,Rio Negro,Convencional,INUMET,POINT (-57.16 -32.51),8.0,paso_de_los_mellizos
2,-32.13,-54.89,Cuchilla Caragauta N,Tacuarembó,Convencional,INUMET,POINT (-54.89 -32.13),4.0,paso_aguiar
3,-31.77,-55.69,Cuchilla del Ombu,Tacuarembó,Convencional,INUMET,POINT (-55.69 -31.77),10.0,paso_del_borracho
4,-31.71,-56.00,El Molino,Tacuarembó,Convencional,INUMET,POINT (-56 -31.71),15.0,tacuarembo
5,-33.52,-56.41,Goñi,Florida,Convencional,INUMET,POINT (-56.41 -33.52),0.0,durazno
6,-31.57,-55.47,Minas de Corrales,Rivera,Convencional,INUMET,POINT (-55.47 -31.57),2.0,paso_de_la_compania
7,-31.57,-55.47,Minas de Corrales,Rivera,Convencional,INUMET,POINT (-55.47 -31.57),10.0,paso_del_borracho
8,-31.60,-54.97,Moirones,Rivera,Convencional,INUMET,POINT (-54.97 -31.6),11.0,picada_de_coelho
9,-33.50,-57.79,Palmitas,Soriano,Convencional,INUMET,POINT (-57.79 -33.5),5.0,bequelo


In [4]:
precipitation_file=pd.read_excel(UYPRECIP_DIR / f"{df.loc[0, 'Estacion']} 1981-2024.xlsx", parse_dates=["Fecha"])
precipitation_file

,Estación,Cod. Pluvio,Fecha,[mm],Comentario
0,Cap. del Sauce,2309,1981-01-01,0.0,NaN
1,Cap. del Sauce,2309,1981-01-02,65.0,NaN
2,Cap. del Sauce,2309,1981-01-03,0.0,NaN
3,Cap. del Sauce,2309,1981-01-04,0.0,NaN
4,Cap. del Sauce,2309,1981-01-05,27.0,NaN
...,...,...,...,...,...
16054,Cap. del Sauce,2309,2024-12-15,0.0,NaN
16055,Cap. del Sauce,2309,2024-12-16,0.0,NaN
16056,Cap. del Sauce,2309,2024-12-17,0.0,NaN
16057,Cap. del Sauce,2309,2024-12-18,0.0,NaN


In [5]:
start_date = pd.to_datetime("1989-01-01")
end_date = pd.to_datetime("2019-12-31")

In [6]:
precipitation_file=precipitation_file[(precipitation_file["Fecha"] >= start_date) & (precipitation_file["Fecha"] <= end_date)].set_index("Fecha")
precipitation_file.head()

,Estación,Cod. Pluvio,[mm],Comentario
Fecha,,,,
1989-01-01,Cap. del Sauce,2309,0.0,NaN
1989-01-02,Cap. del Sauce,2309,0.0,NaN
1989-01-03,Cap. del Sauce,2309,0.0,NaN
1989-01-04,Cap. del Sauce,2309,0.0,NaN
1989-01-05,Cap. del Sauce,2309,0.0,NaN


In [7]:
expected_index = pd.date_range(start=start_date, end=end_date, freq="D")

missing_dates = expected_index.difference(precipitation_file.index)

print(missing_dates)

DatetimeIndex([], dtype='datetime64[ns]', freq='D')


In [8]:
# Amount of missing data

n_missing_flags = (precipitation_file["Comentario"] == "Dato faltante.").sum()

print(n_missing_flags)

59


In [9]:
# Boolean mask for missing
missing = precipitation_file["Comentario"] == "Dato faltante."

# Create groups whenever True/False changes
groups = (missing != missing.shift()).cumsum()

# Compute length of each consecutive missing streak
streak_lengths = missing.groupby(groups).sum()

# Keep only actual missing streaks (length > 0)
streak_lengths = streak_lengths[streak_lengths > 0]

# Count how many streaks of each length exist
streak_distribution = streak_lengths.value_counts().sort_index()

print(streak_distribution)


Comentario
1    53
2     3
Name: count, dtype: int64


In [15]:
for estacion in df["Estacion"].unique():
    print(f"---------{estacion}---------")

    if estacion == "Cañada Grande":
        estacion = "Canada Grande"
    
    if estacion == "Cuchilla Caragauta N":
        estacion = "Cuchilla Caraguata Norte"

    if estacion == "Goñi":
        estacion = "Goni"

    if estacion == "Paso de la Cruz_RN":
        estacion = "Paso de la Cruz_rn"

    if estacion == "Sarandi del Yi":
        continue

    if estacion == "Sarandi Grande":
        continue

    if estacion == "Cuchilla del Ombu":
        precipitation_file=pd.read_excel(UYPRECIP_DIR / f"{estacion} 1982-2024.xlsx", parse_dates=["Fecha"])
    elif estacion == "Tranqueras":
        precipitation_file=pd.read_excel(UYPRECIP_DIR / f"Tranqueras_1982_2024.xlsx", parse_dates=["Fecha"])
    elif estacion == "Vichadero":
        precipitation_file=pd.read_excel(UYPRECIP_DIR / f"Vichadero_1981_2024.xlsx", parse_dates=["Fecha"])
    else:
        precipitation_file=pd.read_excel(UYPRECIP_DIR / f"{estacion} 1981-2024.xlsx", parse_dates=["Fecha"])

    precipitation_file=precipitation_file[(precipitation_file["Fecha"] >= start_date) & (precipitation_file["Fecha"] <= end_date)].set_index("Fecha")

    # Missing dates
    expected_index = pd.date_range(start=start_date, end=end_date, freq="D")
    missing_dates = expected_index.difference(precipitation_file.index)
    print("Missing dates:")
    print(missing_dates)

    # Amount of missing dates
    if estacion == "Tranqueras" or estacion == "Vichadero":
        n_missing_flags = (precipitation_file["ComentarioR3[-]"] == "Dato faltante.").sum()
    else:
        n_missing_flags = (precipitation_file["Comentario"] == "Dato faltante.").sum()
    print("Missing precipitation meassurements:")
    print(n_missing_flags)

    #Distribution of missing streak lengths

    # Boolean mask for missing
    if estacion == "Tranqueras" or estacion == "Vichadero":
        missing = precipitation_file["ComentarioR3[-]"] == "Dato faltante."
    else:
        missing = precipitation_file["Comentario"] == "Dato faltante."

    # Create groups whenever True/False changes
    groups = (missing != missing.shift()).cumsum()

    # Compute length of each consecutive missing streak
    streak_lengths = missing.groupby(groups).sum()

    # Keep only actual missing streaks (length > 0)
    streak_lengths = streak_lengths[streak_lengths > 0]

    # Count how many streaks of each length exist
    streak_distribution = streak_lengths.value_counts().sort_index()

    print("Distribution of missing streak lengths:")
    print(streak_distribution)
    

---------Capilla del Sauce---------
Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Missing precipitation meassurements:
59
Distribution of missing streak lengths:
Comentario
1    53
2     3
Name: count, dtype: int64
---------Cañada Grande---------
Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Missing precipitation meassurements:
134
Distribution of missing streak lengths:
Comentario
1    126
2      4
Name: count, dtype: int64
---------Cuchilla Caragauta N---------
Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Missing precipitation meassurements:
46
Distribution of missing streak lengths:
Comentario
1    44
2     1
Name: count, dtype: int64
---------Cuchilla del Ombu---------
Missing dates:
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Missing precipitation meassurements:
71
Distribution of missing streak lengths:
Comentario
1    63
2     4
Name: count, dtype: int64
---------El Molino---------
Missing dates:
DatetimeInd

In [ ]:
# ver todos los archivos para ver que los comentarios sean iguales
# que hizo horudour para mas de un dia consecutivo